# Notebook 08 — LLM-based Classification (Section 5)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import classification_report, accuracy_score, f1_score

OUTPUTS_DIR = Path("outputs")
MODELS_DIR = Path("models")

print("Libraries loaded.")

Libraries loaded.


## 1. Load Sample Data

In [2]:
test_df = pd.read_csv(OUTPUTS_DIR / 'test.csv').dropna(subset=['text', 'sentiment', 'star_rating', 'category'])
# Use 200-review sample
sample_df = test_df.sample(200, random_state=42).reset_index(drop=True)
print(f"Sample: {len(sample_df)} reviews")
print(sample_df[['text', 'sentiment', 'star_rating', 'category']].head(3))

Sample: 200 reviews
                                                text sentiment  star_rating  \
0  Hello\nvery unhappy with this insurance I woul...  negative            1   
1  I am satisfied with the service. Very welcome,...  positive            5   
2  Simple, fast and the very advantageous price !...  positive            5   

           category  
0           Pricing  
1  Customer Service  
2           Pricing  


## 2. Zero-Shot Classification with HuggingFace (No API Key Required)

In [3]:
from transformers import pipeline

# Zero-shot for sentiment
print("Loading zero-shot classifier for sentiment...")
zs_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)

SENTIMENT_LABELS = ['negative', 'neutral', 'positive']
CATEGORY_LABELS = ['Pricing', 'Coverage', 'Enrollment', 'Customer Service', 'Claims Processing', 'Cancellation']

Loading zero-shot classifier for sentiment...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [4]:
from tqdm import tqdm

def zero_shot_batch(texts, labels, batch_size=16):
    results = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        preds = zs_classifier(batch, labels, multi_label=False)
        results.extend([p['labels'][0] for p in preds])
    return results

print("Zero-shot sentiment classification on 200 samples...")
zs_sentiment_preds = zero_shot_batch(sample_df['text'].tolist(), SENTIMENT_LABELS)

Zero-shot sentiment classification on 200 samples...


100%|██████████| 13/13 [01:24<00:00,  6.47s/it]


In [5]:
print("Zero-shot category classification on 200 samples...")
zs_category_preds = zero_shot_batch(sample_df['text'].tolist(), CATEGORY_LABELS)

Zero-shot category classification on 200 samples...


100%|██████████| 13/13 [02:36<00:00, 12.07s/it]


In [6]:
# Evaluate zero-shot sentiment
y_true_sent = sample_df['sentiment'].astype(str).tolist()
y_pred_sent = zs_sentiment_preds

acc_sent = accuracy_score(y_true_sent, y_pred_sent)
f1_sent = f1_score(y_true_sent, y_pred_sent, average='macro', zero_division=0)
print(f"Zero-Shot Sentiment | Accuracy: {acc_sent:.4f} | F1-macro: {f1_sent:.4f}")
print(classification_report(y_true_sent, y_pred_sent, zero_division=0))

Zero-Shot Sentiment | Accuracy: 0.7500 | F1-macro: 0.5505
              precision    recall  f1-score   support

    negative       0.78      0.92      0.84        87
     neutral       0.00      0.00      0.00        34
    positive       0.74      0.89      0.81        79

    accuracy                           0.75       200
   macro avg       0.51      0.60      0.55       200
weighted avg       0.63      0.75      0.69       200



In [7]:
# Evaluate zero-shot category
y_true_cat = sample_df['category'].astype(str).tolist()
y_pred_cat = zs_category_preds

acc_cat = accuracy_score(y_true_cat, y_pred_cat)
f1_cat = f1_score(y_true_cat, y_pred_cat, average='macro', zero_division=0)
print(f"Zero-Shot Category | Accuracy: {acc_cat:.4f} | F1-macro: {f1_cat:.4f}")
print(classification_report(y_true_cat, y_pred_cat, zero_division=0))

Zero-Shot Category | Accuracy: 0.3950 | F1-macro: 0.2990
                   precision    recall  f1-score   support

     Cancellation       0.23      0.64      0.34        11
Claims Processing       0.25      0.10      0.14        10
         Coverage       0.41      0.50      0.45        58
 Customer Service       0.39      0.58      0.47        38
       Enrollment       0.00      0.00      0.00         4
          Pricing       0.87      0.25      0.39        79

         accuracy                           0.40       200
        macro avg       0.36      0.34      0.30       200
     weighted avg       0.56      0.40      0.40       200



## 3. Few-Shot Prompting with a Local LLM via Ollama (Optional)

In [11]:
# This section requires Ollama to be installed and running
# Install: curl https://ollama.ai/install.sh | sh
# Run: ollama pull llama3.2
# Then: ollama serve

import subprocess
import json
import requests

def check_ollama():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        return r.status_code == 200
    except:
        return False

OLLAMA_AVAILABLE = check_ollama()
print(f"Ollama available: {OLLAMA_AVAILABLE}")
if not OLLAMA_AVAILABLE:
    print("Ollama not running. Skipping few-shot LLM section.")
    print("To enable: install Ollama, run 'ollama pull llama3.2', then 'ollama serve'")

Ollama available: True


In [12]:
if OLLAMA_AVAILABLE:
    def ollama_classify(text, task='sentiment'):
        if task == 'sentiment':
            labels = "negative, neutral, or positive"
            examples = """Review: \"The claims process was terrible and slow.\" -> negative
Review: \"Average service, nothing special.\" -> neutral
Review: \"Fantastic support team, highly recommend!\" -> positive"""
        else:
            labels = "Pricing, Coverage, Enrollment, Customer Service, Claims Processing, or Cancellation"
            examples = """Review: \"The premium is too expensive.\" -> Pricing
Review: \"Filing a claim was very easy.\" -> Claims Processing
Review: \"Canceling my policy was difficult.\" -> Cancellation"""
        
        prompt = f"""Classify the following insurance review.
{examples}

Review: \"{text[:200]}\"
Answer with only the label ({labels}):"""
        
        response = requests.post("http://localhost:11434/api/generate",
            json={"model": "llama3.2", "prompt": prompt, "stream": False},
            timeout=30
        )
        return response.json().get('response', '').strip().lower()
    
    print("Testing Ollama few-shot classification...")
    test_review = sample_df['text'].iloc[0]
    print(f"Review: {test_review[:100]}")
    print(f"Predicted sentiment: {ollama_classify(test_review, 'sentiment')}")
    
    # Run on first 50 samples
    llm_sentiment = [ollama_classify(t, 'sentiment') for t in tqdm(sample_df['text'][:50].tolist())]
    y_true_50 = sample_df['sentiment'][:50].astype(str).tolist()
    
    # Normalize predictions
    def normalize_sentiment(pred):
        pred = pred.lower()
        if 'positive' in pred: return 'positive'
        if 'negative' in pred: return 'negative'
        return 'neutral'
    
    llm_sentiment_norm = [normalize_sentiment(p) for p in llm_sentiment]
    acc_llm = accuracy_score(y_true_50, llm_sentiment_norm)
    f1_llm = f1_score(y_true_50, llm_sentiment_norm, average='macro', zero_division=0)
    print(f"\nOllama few-shot Sentiment (50 samples) | Acc: {acc_llm:.4f} | F1: {f1_llm:.4f}")
else:
    print("Skipped — Ollama not available.")
    acc_llm = None
    f1_llm = None

Testing Ollama few-shot classification...
Review: Hello
very unhappy with this insurance I would not recommend it in no way
I phone every day followin
Predicted sentiment: negative


100%|██████████| 50/50 [00:18<00:00,  2.75it/s]


Ollama few-shot Sentiment (50 samples) | Acc: 0.6800 | F1: 0.5743


## 4. Save LLM Results

In [13]:
sample_df['pred_sentiment_zeroshot'] = zs_sentiment_preds
sample_df['pred_category_zeroshot'] = zs_category_preds

sample_df.to_csv(OUTPUTS_DIR / 'llm_predictions.csv', index=False)

results = pd.DataFrame([
    {'model': 'ZeroShot-BART', 'task': 'sentiment', 'accuracy': acc_sent, 'f1_macro': f1_sent},
    {'model': 'ZeroShot-BART', 'task': 'category', 'accuracy': acc_cat, 'f1_macro': f1_cat},
])
if acc_llm is not None:
    results = pd.concat([results, pd.DataFrame([
        {'model': 'Ollama-LLaMA3.2', 'task': 'sentiment', 'accuracy': acc_llm, 'f1_macro': f1_llm}
    ])])

results.to_csv(OUTPUTS_DIR / 'results_llm.csv', index=False)
print("Saved llm_predictions.csv and results_llm.csv")
print(results)

Saved llm_predictions.csv and results_llm.csv
             model       task  accuracy  f1_macro
0    ZeroShot-BART  sentiment     0.750  0.550451
1    ZeroShot-BART   category     0.395  0.299029
0  Ollama-LLaMA3.2  sentiment     0.680  0.574346


## Summary
- Zero-shot sentiment + category classification with `facebook/bart-large-mnli`
- Optional few-shot prompting with local Ollama (llama3.2)
- Evaluated on 200-review sample
- Results saved to `outputs/results_llm.csv`